# Identity, Isolation, and Compliance

A Riverside editor signs in successfully and asks for a manuscript. That is only the start of the decision. The assistant still needs to prove the editor belongs to the tenant, is active, has the requested role, is in an allowed region, declared an allowed purpose, and is assigned to the title.

In this notebook you will predict `ALLOW` or `DENY` before each run, then find the first boundary that should stop the request. The cases are synthetic. A local pass shows that the teaching code behaved as expected; it does not prove a deployed control or a compliance conclusion.

**Keep this picture in mind:** sign-in identifies the caller; trusted policy decides what that caller may see or do.


## 0 - Riverside's identity problem

A disabled contractor still appears in a nested editor group. In another request, an EU editor changes the request fields to look like a US editor. If Riverside trusts those fields, both callers can cross a boundary they should not cross.

```mermaid
flowchart LR
    A["Signed-in caller"] --> B["Request claims tenant,<br/>role, region, title"]
    B --> C{"Compare with trusted<br/>identity and policy"}
    C -->|Mismatch, inactive, or unknown| D["DENY<br/>record reason<br/>stop"]
    C -->|All checks pass| E["Continue with bounded context"]
```

**Predict before running:** Which should be checked first for the contractor: the stale editor group or the active/disabled status? Write one sentence explaining why.

The safe answer is fail closed: missing, stale, conflicting, or untrusted context is a denial, not a best guess.


### What the notebook can and cannot prove

| Label | Plain meaning | Riverside example |
|---|---|---|
| `[Local-static]` | We read source or expected results; nothing ran | The fixture contains a cross-tenant deny case |
| `[Local-measured]` | A named local run produced a result | This run matched all expected fixture decisions |
| `[Modeled]` | Policy says how the design should behave | EU manuscript requests should remain in `REG-UKS` |
| `[External validation required]` | An authorized owner must test the real environment | Deployed identity revocation, RBAC, cache, network, and logs preserve the boundary |

A green cell is not a security or compliance attestation. Record the environment, source commit, fixture version, date, result, and limits before calling a result `[Local-measured]`.

**Plain takeaway:** this notebook proves local mechanics only. Production and legal claims stay open until their owners supply evidence.


In [ ]:
# -- Load frozen and local synthetic contracts ----------------------------
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
import hashlib
import json


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "AUTHORING_GUIDE.md").is_file() and (candidate / "learning" / "role-based-tracks" / "fde" / "shared").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook inside ai-portfolio.")


def load_json(path: Path) -> dict[str, Any]:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


ROOT = find_repo_root(Path.cwd().resolve())
CHAPTER = ROOT / "learning" / "role-based-tracks" / "fde" / "04-identity-isolation-and-compliance"
SHARED = ROOT / "learning" / "role-based-tracks" / "fde" / "shared" / "fixtures"
engagement = load_json(SHARED / "riverside-engagement-v1.json")
source_samples = load_json(SHARED / "riverside-source-samples-v1.json")
expected_facts = load_json(SHARED / "expected-facts-v1.json")
local_fixture = load_json(CHAPTER / "fixtures" / "identity-scenarios-v1.json")
fde04_facts = [fact for fact in expected_facts["facts"] if "FDE-04" in fact["notebook_ids"]]

print(f"LOCAL-STATIC: frozen={engagement['fixture_version']}, scenarios={len(local_fixture['scenarios'])}")
print(f"  FDE-04 expected facts: {len(fde04_facts)}")
print("  no network, IdP, model, vector store, tool, or cloud service was contacted")

## 1 - First failure: the request lies about authority

Riverside receives a request from an EU editor whose body says: US tenant, US region, broader role, and a US title. A shortcut implementation reads those fields and returns the US manuscript.

```mermaid
flowchart LR
    A["EU editor identity"] --> B["Forged US request fields"]
    B --> C["Request-only check"]
    C --> D["Wrong: US resource returned"]
    A --> E["Trusted identity record"]
    E --> F["Mismatch found"]
    F --> G["DENY and audit"]
```

**Predict before running:** Will the intentionally unsafe check allow, deny, or error? What fact is it missing?

| Before | After |
|---|---|
| Sign-in succeeds, so request fields are treated as facts | Sign-in identifies the actor; trusted records supply tenant, active roles, region, and titles |
| Forged scope reaches retrieval | A mismatch becomes a named deny before retrieval |

**Plain takeaway:** prompts and request fields cannot grant authority. Rebuild context from trusted data.


In [ ]:
# -- Expose the request-only authorization defect -------------------------
resources = {item["resource_id"]: item for item in local_fixture["resources"]}
scenarios = {item["scenario_id"]: item for item in local_fixture["scenarios"]}


def unsafe_request_only_authorized(request: dict[str, Any], resource: dict[str, Any]) -> bool:
    return (
        request["tenant_id"] == resource["tenant_id"]
        and request["region_id"] == resource["region_id"]
        and bool(set(request["role_ids"]) & set(resource["allowed_role_ids"]))
    )


tampered = dict(scenarios["ISO-RIV-002"]["request"])
tampered.update({"tenant_id": "TEN-RIV-US", "region_id": "REG-EUS", "title_ids": ["TITLE-HARBOR"]})
unsafe_result = unsafe_request_only_authorized(tampered, resources["RES-RIV-US-MANUSCRIPT-HARBOR"])
print(f"UNSAFE EXPECTATION: request-only check returns {unsafe_result}")
print("  actor identity never participated in the decision")

## 2 - Build trusted context and stop on the first failed check

The gateway looks up the actor, confirms the identity is active, checks tenant and region, requires a purpose and assigned title, then compares requested roles with active trusted roles. The request may ask for less authority than the caller has. It may never ask for more.

```mermaid
flowchart TD
    A["Riverside request"] --> B{"Identity active?"}
    B -->|No or unknown| X["DENY: identity_disabled"]
    B -->|Yes| C{"Tenant and region allowed?"}
    C -->|No| Y["DENY"]
    C -->|Yes| D{"Purpose and title allowed?"}
    D -->|No| Z["DENY"]
    D -->|Yes| E{"Requested roles all trusted?"}
    E -->|No| W["DENY: role_escalation"]
    E -->|Yes| F["Trusted request context"]
```

**Predict before running:** The disabled contractor still has a stale editor group. Does the group rescue the request? Name the exact first denial reason.

Do not silently remove an untrusted requested role and continue. That would hide an escalation attempt. Deny it and record why.

**Plain takeaway:** active status comes before role membership, and any missing or conflicting required field fails closed.


In [ ]:
# -- Normalize trusted request context ------------------------------------
constraints = engagement["identity_and_data_constraints"]
required_context = tuple(constraints["required_request_context"])
allowed_purposes = frozenset(constraints["purposes"])
tenant_by_id = {item["tenant_id"]: item for item in constraints["tenants"]}
identity_records = {
    record["payload"]["actor_id"]: record["payload"]
    for record in source_samples["records"]
    if record["payload_type"] == "identity_record"
}


class AuthorizationDenied(Exception):
    def __init__(self, reason: str, boundary: str) -> None:
        super().__init__(reason)
        self.reason = reason
        self.boundary = boundary


@dataclass(frozen=True)
class RequestContext:
    tenant_id: str
    actor_id: str
    role_ids: tuple[str, ...]
    region_id: str
    purpose: str
    title_ids: tuple[str, ...]
    trace_id: str


def active_roles(identity: dict[str, Any]) -> frozenset[str]:
    return frozenset(identity.get("role_ids", identity.get("direct_role_ids", [])))


def normalize_context(request: dict[str, Any]) -> RequestContext:
    missing = [field for field in required_context if field not in request or request[field] in (None, "", [])]
    if missing:
        raise AuthorizationDenied("missing_context", "gateway")
    identity = identity_records.get(request["actor_id"])
    if identity is None or not identity.get("enabled", False):
        raise AuthorizationDenied("identity_disabled", "gateway")
    if request["tenant_id"] not in identity["tenant_ids"]:
        raise AuthorizationDenied("tenant_not_entitled", "gateway")
    tenant = tenant_by_id.get(request["tenant_id"])
    if tenant is None or request["region_id"] not in tenant["allowed_region_ids"]:
        raise AuthorizationDenied("region_not_allowed", "gateway")
    if request["purpose"] not in allowed_purposes:
        raise AuthorizationDenied("purpose_not_allowed", "gateway")
    trusted_roles = active_roles(identity)
    requested_roles = frozenset(request["role_ids"])
    if not requested_roles.issubset(trusted_roles):
        raise AuthorizationDenied("role_escalation", "gateway")
    requested_titles = frozenset(request["title_ids"])
    if not requested_titles.issubset(frozenset(identity.get("title_ids", []))):
        raise AuthorizationDenied("title_not_assigned", "gateway")
    return RequestContext(request["tenant_id"], request["actor_id"], tuple(sorted(requested_roles)), request["region_id"], request["purpose"], tuple(sorted(requested_titles)), request["trace_id"])


print(f"LOCAL-STATIC: required fields={required_context}")

## 3 - Protect retrieval twice

Suppose the gateway approves an EU editor for one assigned title. Riverside must still stop a US manuscript from entering the model if an index adapter drops a filter. Use two controls:

1. Send mandatory tenant, region, purpose, title, role, classification, and active-state filters before search.
2. Check every returned record again before it can enter model context.

```mermaid
flowchart LR
    A["Trusted context"] --> B["Mandatory server filter"]
    B --> C["Search"]
    C --> D["Returned candidates"]
    D --> E{"Independent record check"}
    E -->|Match| F["Authorized evidence"]
    E -->|Mismatch| G["Drop, audit, investigate"]
```

**Predict before running:** If the adapter returns a high-scoring US manuscript to an EU editor, which control catches it? Why is prompt text not enough?

Post-checking alone is too late if unauthorized content already crossed into model context. Filtering alone is too fragile if a backend, cache, or adapter drops a condition.

**Plain takeaway:** filter before search and verify after search. Both checks deny on uncertainty.


In [ ]:
# -- Build retrieval filters and verify returned resources ----------------
def retrieval_filter(context: RequestContext) -> dict[str, Any]:
    return {
        "tenant_id": context.tenant_id,
        "region_id": context.region_id,
        "purpose": context.purpose,
        "title_ids": list(context.title_ids),
        "role_ids": list(context.role_ids),
        "deletion_state": "active",
    }


def authorize_resource(context: RequestContext, resource: dict[str, Any]) -> None:
    if resource["tenant_id"] != context.tenant_id:
        raise AuthorizationDenied("tenant_mismatch", "retrieval")
    if resource["region_id"] != context.region_id:
        raise AuthorizationDenied("region_mismatch", "retrieval")
    if resource["deletion_state"] != "active":
        raise AuthorizationDenied("resource_inactive", "retrieval")
    if context.purpose not in resource["allowed_purposes"]:
        raise AuthorizationDenied("purpose_mismatch", "retrieval")
    if resource["title_id"] is not None and resource["title_id"] not in context.title_ids:
        raise AuthorizationDenied("title_not_assigned", "retrieval")
    if not set(context.role_ids) & set(resource["allowed_role_ids"]):
        raise AuthorizationDenied("role_not_authorized", "retrieval")


print("LOCAL-STATIC: filter uses tenant, region, purpose, title, role, and active state")

## 4 - Authorize the exact tool, not the story around it

A Riverside editor asks the assistant to change contract rights. The identity is valid and a person clicks Confirm. The action is still prohibited. Human approval can satisfy a confirmation rule for an otherwise allowed action; it cannot create a permission the actor does not have.

```mermaid
flowchart TD
    A["Exact tool + payload"] --> B{"Known schema and target?"}
    B -->|No| X["DENY and audit"]
    B -->|Yes| C{"Role and purpose allow it?"}
    C -->|No| X
    C -->|Yes| D{"Action prohibited?"}
    D -->|Yes| X
    D -->|No| E{"Required approval present?"}
    E -->|No| Y["Ask for bounded approval"]
    E -->|Yes| F["Use minimum-scope service identity"]
```

**Predict before running:** For `ISO-RIV-008`, does confirmation permit `rights.change_contract`? Answer before viewing the result.

**Before:** model intent or a broad confirmation can be mistaken for permission.

**After:** deterministic policy checks the exact tool, target, payload, role, purpose, prohibited actions, and approval requirement. Unknown tools and unknown policy fail closed.


In [ ]:
# -- Authorize concrete tools outside model reasoning ---------------------
tool_policies = {item["tool_name"]: item for item in local_fixture["tool_policies"]}


def authorize_tool(context: RequestContext, request: dict[str, Any]) -> None:
    policy = tool_policies.get(request.get("tool_name"))
    if policy is None:
        raise AuthorizationDenied("unknown_tool", "tool")
    if not set(context.role_ids) & set(policy["allowed_role_ids"]):
        raise AuthorizationDenied("tool_not_authorized", "tool")
    if context.purpose not in policy["allowed_purposes"]:
        raise AuthorizationDenied("tool_purpose_mismatch", "tool")
    if policy["prohibited_actions"]:
        raise AuthorizationDenied("prohibited_action", "tool")
    if policy["requires_human_confirmation"] and not request["human_confirmed"]:
        raise AuthorizationDenied("human_confirmation_required", "tool")


print("LOCAL-STATIC: human confirmation never expands the allowed-role set")

## 5 - Keep enough audit evidence, but do not create a new leak

After an allow or deny, Riverside needs to answer two different questions:

- **Restricted audit:** Why did this exact request stop or continue?
- **Metrics:** How often do broad decision types occur?

The restricted audit may hold protected references needed for an investigation. Metric labels must not contain actor, tenant, trace, request, document, prompt, or completion IDs. The public response must not echo roles, filters, token claims, or backend errors.

```mermaid
flowchart LR
    A["Decision + trusted context"] --> B["Restricted audit"]
    A --> C["Allowlisted low-cardinality metrics"]
    A --> D["Internal response assembly"]
    D --> E["Minimal answer or refusal"]
```

**Predict before running:** Which destination may retain a protected actor reference and trace ID: restricted audit, metric labels, or the public response?

Hashing an identity does not make it anonymous. Production retention, access, deletion exceptions, and legal holds still need Security and Legal/Privacy approval.

**Plain takeaway:** record the decision without copying content, and expose less in the response than the system used internally.


In [ ]:
# -- Create content-free audit and minimized public response --------------
def stable_fingerprint(value: str) -> str:
    return hashlib.sha256(f"riverside-synthetic-audit:{value}".encode()).hexdigest()[:16]


def audit_event(context: RequestContext | None, request: dict[str, Any], decision: str, reason: str, boundary: str) -> dict[str, Any]:
    return {
        "event_type": "authorization_decision",
        "actor_ref": stable_fingerprint(request["actor_id"]),
        "tenant_ref": stable_fingerprint(request["tenant_id"]),
        "trace_id": request["trace_id"],
        "region_id": request["region_id"],
        "purpose": request["purpose"] or "missing",
        "effective_role_ids": list(context.role_ids) if context else [],
        "boundary": boundary,
        "decision": decision,
        "reason": reason,
        "policy_version": local_fixture["fixture_version"],
        "contains_customer_content": False,
        "contains_credentials": False,
    }


def assemble_response(context: RequestContext | None, decision: str, reason: str) -> tuple[dict[str, Any], dict[str, Any]]:
    internal = {
        "decision_context": asdict(context) if context else None,
        "authorization": {"decision": decision, "reason": reason},
    }
    public = {
        "object": "chat.completion",
        "message": "Authorized synthetic response." if decision == "allow" else "I cannot complete that request.",
        "refusal": None if decision == "allow" else "authorization_denied",
        "citations": [],
        "trace": {"trace_id": context.trace_id if context else "redacted-denial-trace"},
    }
    return internal, public


print("LOCAL-STATIC: internal assembly carries context; public response minimizes it")

## 6 - Run the complete Riverside decision path

The next code cell sends all nine synthetic cases through gateway, retrieval or tool policy, audit, and response assembly. A denial must stop downstream work.

```mermaid
flowchart LR
    A["Scenario"] --> B["Gateway checks"]
    B -->|Denied| C["Audit reason + refusal"]
    B -->|Resource request| D["Filter + verify"]
    B -->|Tool request| E["Exact tool policy"]
    D --> F["Audit + minimal response"]
    E --> F
```

**Predict before running:** Expect two bounded allows and seven denies. Which case would be the most serious false allow, and which boundary should stop it?

**Pass condition:** every observed decision and denial reason matches the fixture, with zero false allows. Any forbidden allow is stop-ship.


In [ ]:
# -- Compose the deterministic decision path ------------------------------
def evaluate_scenario(scenario: dict[str, Any]) -> dict[str, Any]:
    request = scenario["request"]
    context: RequestContext | None = None
    decision, reason, boundary = "deny", "uninitialized", "gateway"
    try:
        context = normalize_context(request)
        if request["resource_id"] is not None:
            authorize_resource(context, resources[request["resource_id"]])
            boundary = "retrieval"
        elif request["tool_name"] is not None:
            authorize_tool(context, request)
            boundary = "tool"
        else:
            raise AuthorizationDenied("missing_target", "gateway")
        decision, reason = "allow", "authorized"
    except AuthorizationDenied as error:
        decision, reason, boundary = "deny", error.reason, error.boundary
    audit = audit_event(context, request, decision, reason, boundary)
    internal, public = assemble_response(context, decision, reason)
    return {
        "scenario_id": scenario["scenario_id"],
        "decision": decision,
        "reason": reason,
        "boundary": boundary,
        "matches_expected": decision == scenario["expected_decision"] and reason == scenario["expected_reason"],
        "audit": audit,
        "internal_response": internal,
        "public_response": public,
    }


# This cell is intentionally unexecuted in the committed notebook.
results = [evaluate_scenario(item) for item in local_fixture["scenarios"]]
matched = sum(item["matches_expected"] for item in results)
false_allows = [
    item["scenario_id"]
    for item, case in zip(results, local_fixture["scenarios"])
    if case["expected_decision"] == "deny" and item["decision"] == "allow"
]
for item in results:
    print(f"{item['scenario_id']}: {item['decision']}/{item['reason']} at {item['boundary']} - expected match={item['matches_expected']}")
print(f"LOCAL-MEASURED ONLY AFTER EXECUTION: {matched}/{len(results)} expected decisions")
print(f"  false allows: {false_allows}")
print("  does not prove cloud, customer, residency, privacy, or legal controls")

### Change one thing and predict the result

Run the next cell once with its default synthetic request. Then change only one field at a time:

- add an untrusted role;
- switch tenant;
- remove purpose;
- switch region;
- request an unassigned title.

Before each run, write `ALLOW` or `DENY` and name the first boundary that should decide. Do not add production identifiers.

After each run, confirm:

1. A deny stopped downstream work and produced an audit reason.
2. An allow used only active trusted roles and assigned titles.
3. Retrieval filtered before search and verified after search.
4. Confirmation did not broaden tool authority.
5. The public response omitted token claims, roles, and filters.


In [ ]:
# -- Change one synthetic authorization dimension -------------------------
exercise = dict(scenarios["ISO-RIV-001"]["request"])
# CHANGE THIS: try TEN-RIV-US, REG-EUS, empty purpose, or ROLE-RIGHTS-COUNSEL.
exercise["role_ids"] = ["ROLE-EDITOR"]
exercise_case = {
    "scenario_id": "ISO-RIV-YOUR-TURN",
    "request": exercise,
    "expected_decision": "allow",
    "expected_reason": "authorized",
}
exercise_result = evaluate_scenario(exercise_case)
print(json.dumps({key: exercise_result[key] for key in ("decision", "reason", "boundary")}, indent=2))

## 7 - Turn the code into a review package

A reviewer should not have to reconstruct the Riverside control story from one large document. Use six connected views:

```mermaid
flowchart TD
    A["Identity flow:<br/>where context changes"] --> G["Isolation review"]
    B["RBAC matrix:<br/>who may do what"] --> G
    C["Residency map:<br/>where data moves"] --> G
    D["Threat sketch:<br/>how boundaries fail"] --> G
    E["Controls matrix:<br/>control, owner, evidence"] --> G
    F["Isolation report:<br/>expected vs observed"] --> G
    G --> H["External gates remain open"]
```

| Riverside question | Local answer | Still needed outside |
|---|---|---|
| Did the synthetic stale contractor get denied? | Recorded local negative test | IdP revocation and gateway test in the target environment |
| Did the synthetic cross-tenant result get blocked? | Local filter and post-check | Index, cache, and adapter negative tests |
| Did metrics avoid identity IDs? | Schema/source inspection | Samples from real failure and load paths |
| Is the service compliant? | The notebook cannot answer this | Authorized security, privacy, legal, and customer review for a named scope |

Do not paste production logs, tokens, or customer content into these artifacts. Store approved references and redacted decision metadata. Keep `NOT RUN` wherever no execution occurred.

**Plain takeaway:** a completed document is not an effective control. Every control needs an owner, evidence, status, and revalidation trigger.


In [ ]:
# ── Inventory committed artifacts without changing them ─────────────────
artifact_names = [
    "identity-flow.md",
    "rbac-matrix.md",
    "data-flow-residency-map.md",
    "threat-model.md",
    "controls-matrix.md",
    "isolation-test-report.md",
    "notebook-output-record.md",
]
artifact_paths = [CHAPTER / "templates" / name for name in artifact_names]
missing_artifacts = [str(path) for path in artifact_paths if not path.exists()]
assert not missing_artifacts, missing_artifacts
for path in artifact_paths:
    print(f"LOCAL-STATIC: {path.name} is present")
print("  presence is not review quality or control effectiveness")

## 8 - What Riverside has now, and what remains open

```mermaid
flowchart LR
    A["Untrusted request claims"] --> B["Trusted context"]
    B --> C["Filter + verify"]
    C --> D["Exact tool policy"]
    D --> E["Protected audit"]
    E --> F["Minimal response"]
    F --> G["Local run record"]
    G --> H["Production validation"]
```

| Riverside risk | Before | After this local chapter |
|---|---|---|
| Stale contractor | A nested group could look current | Inactive identity denies before role checks |
| Cross-tenant retrieval | Caller filters could select another tenant | Trusted server filter plus independent result check |
| Tool overreach | Confirmation could be mistaken for authority | Exact action and payload policy; prohibited action still denies |
| Evidence overclaim | Green fixtures could be called compliant | Local, modeled, and external evidence stay separate |

### Plain takeaways

1. Sign-in identifies a caller; it does not grant a resource or action.
2. Requested authority may shrink trusted authority; it never expands it.
3. Missing, stale, unknown, or conflicting security context fails closed.
4. Retrieval needs a server filter before search and verification after search.
5. Approval cannot make a forbidden tool action legal.
6. Local tests open the production review; they do not close security, privacy, residency, or compliance claims.

Keep `templates/isolation-test-report.md` and `templates/notebook-output-record.md` at `NOT RUN` until an authorized run records the environment, commit, fixture versions, decisions, false allows, false denies, audit evidence, and limits.

**Forward to FDE 05:** regional route, quota, latency, support, and evidence-owner gaps become planning inputs. Identity and isolation failures remain hard release vetoes, never weighted tradeoffs.
